# 03 Embeddings

Companion notebook to the textbook. Run from the `notebooks/` folder inside the project (the code uses `..` paths). Cells marked NEEDS INTERNET download a Hugging Face model on first run.

# Chapter 8: Large Language Models

## 8.0 What an LLM is

## 8.1 Tokens, token IDs and a tiny tokenizer

In [ ]:
from collections import Counter
corpus = ["low"] * 5 + ["lower"] * 2 + ["lowest"] * 3
words = [list(w) for w in corpus]
merges = []
for _ in range(4):
    pairs = Counter()
    for w in words:
        pairs.update(zip(w, w[1:]))
    (a, b), n = pairs.most_common(1)[0]
    merges.append((a + b, n))
    for w in words:
        i = 0
        while i < len(w) - 1:
            if w[i] == a and w[i + 1] == b:
                w[i:i + 2] = [a + b]
            else:
                i += 1
print(merges)
print(words[0], words[5], words[7])
vocab = {tok: i for i, tok in enumerate(sorted({t for w in words for t in w}))}
print(vocab)

## 8.2 How an LLM generates: logits, temperature, top-k and top-p

In [ ]:
import numpy as np
tokens = ["Paris", "London", "banana", "the"]
logits = np.array([4.0, 2.5, 0.5, 1.0])
def probs(l, T=1.0):
    e = np.exp((l - l.max()) / T); return e / e.sum()
for T in (0.2, 1.0, 2.0):
    print(f"T={T}:", {t: round(float(x), 3) for t, x in zip(tokens, probs(logits, T))})
p = probs(logits)
order = np.argsort(-p)
print("top-k=2 keeps:", [tokens[i] for i in order[:2]])
cum = np.cumsum(p[order])
keep = [tokens[i] for i, c in zip(order, cum - p[order]) if c < 0.9]
print("top-p=0.9 keeps:", keep)

## 8.3 Context window, token budget, model size and memory (RAM/VRAM)

In [ ]:
def weights_gb(params_billion, bits):
    return params_billion * 1e9 * bits / 8 / 1e9
for name, p in (("1B", 1), ("3B", 3), ("7B", 7)):
    print(name, {f"{b}-bit": round(weights_gb(p, b), 1) for b in (16, 8, 4)}, "GB (weights only)")
chunk_tokens = 800 / 4          # about 4 characters per token
top_k = 4
print("prompt budget:", int(top_k * chunk_tokens + 150), "tokens for context+question+instructions")

## 8.4 Limitations and hallucinations

## 8.5 Choosing free open-source models

## 8.6 Exercises

## 8.7 Interview questions and answers

# Chapter 9: The Hugging Face Ecosystem

## 9.1 What Hugging Face is

## 9.2 Model cards and licenses

## 9.3 Where models are stored

## 9.4 Using the Pipeline API, tokenizers and models

In [ ]:
# NEEDS INTERNET: downloads a Hugging Face model on first run
from transformers import pipeline, AutoTokenizer
clf = pipeline("sentiment-analysis")            # downloads a small default model
print(clf("This textbook is clear and helpful."))
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
enc = tok("Random forests use many trees.")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"]))

## 9.5 Sentence Transformers: loading and encoding

In [ ]:
# NEEDS INTERNET: downloads a Hugging Face model on first run
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vecs = model.encode(["What is supervised learning?", "Learning from labelled examples."],
                    normalize_embeddings=True)
print(vecs.shape)              # expected: (2, 384)
print(float(vecs[0] @ vecs[1]))   # cosine similarity (a number between -1 and 1)

## 9.6 Choosing a model by hardware and task

## 9.7 Running models locally

## 9.8 Interview questions and answers

# Chapter 10: Embeddings

## 10.1 What embeddings are and why we need them

## 10.2 Embeddings as geometry: similarity and distance

In [ ]:
import numpy as np
# Hand-made 3-d "embeddings" to show the geometry (dimensions: vehicle-ness, fruit-ness, speed)
emb = {"car":        np.array([0.9, 0.0, 0.5]),
       "automobile": np.array([0.85, 0.05, 0.5]),
       "banana":     np.array([0.0, 0.95, 0.1])}
unit = {k: v / np.linalg.norm(v) for k, v in emb.items()}
cos = lambda a, b: float(unit[a] @ unit[b])
print("car~automobile:", round(cos("car", "automobile"), 3))
print("car~banana:    ", round(cos("car", "banana"), 3))
d = np.linalg.norm(unit["car"] - unit["banana"])
print("check identity:", round(d ** 2, 3), "=", round(2 - 2 * cos("car", "banana"), 3))

## 10.3 Our offline fallback embedder: what it can and cannot do

In [ ]:
import sys
sys.path.insert(0, "..")
from src.embeddings import HashingEmbedder
e = HashingEmbedder()
texts = ["a random forest is many decision trees voting",
         "decision trees vote together in a random forest",
         "the car is fast", "the automobile is quick"]
v = e.embed(texts)
print("related wording :", round(float(v[0] @ v[1]), 3))
print("unrelated       :", round(float(v[0] @ v[2]), 3))
print("synonyms only   :", round(float(v[2] @ v[3]), 3))

## 10.4 Real sentence embeddings with a free model

In [ ]:
# NEEDS INTERNET: downloads a Hugging Face model on first run
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = ["The car is fast", "The automobile is quick", "The banana is yellow"]
E = model.encode(texts, normalize_embeddings=True)
print(E.shape)                      # expected (3, 384)
print(np.round(E @ E.T, 2))         # 3x3 cosine similarity matrix
# Expectation to check on your machine: the car/automobile pair should score
# higher than either sentence against the banana sentence.

## 10.5 Choosing an embedding model

## 10.6 Interview questions and answers